In [5]:
# Demo: insert_kmers replaces a zero-length marker <bc/> with <bc>[kmer]</bc>
import poolparty as pp
pp.init()
template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA')\
    .named('template_pool')\
    .print_library()
    
wt_pool = template_pool\
    .repeat_states(3, seq_name_prefix='wt.v')\
    .insert_kmers('bc', length=5)\
    .named('wt_pool')\
    .print_library()

shuf_pool = template_pool\
    .shuffle_seq('cre', mode='hybrid', num_hybrid_states=5, seq_name_prefix='shuff_')\
    .repeat_states(3, seq_name_prefix='v')\
    .named('shuf_pool')
    
mut_pool = template_pool\
    .mutagenize('cre', mutation_rate=0.1, mode='hybrid', num_hybrid_states=5, seq_name_prefix='mut_')\
    .repeat_states(3, seq_name_prefix='v')\
    .insert_kmers('bc', length=5)\
    .named('mut_rate_pool')

del_pool = template_pool\
    .deletion_scan('cre', deletion_length=5, positions=slice(None, None, 5), mode='sequential', seq_name_prefix='del_')\
    .repeat_states(3, seq_name_prefix='v')\
    .insert_kmers('bc', length=5).named('del_pool')

site_pool = pp.from_seqs(['AAAAA', 'CCCCC'], mode='sequential', seq_name_prefix='site_')

repl_pool = template_pool\
    .replacement_scan('cre', ins_pool=site_pool, positions=slice(None, None, 5), mode='sequential', seq_name_prefix='repl_')\
    .repeat_states(3, seq_name_prefix='v')\
    .insert_kmers('bc', length=5)\
    .named('repl_pool')

combo_pool = pp.stack([wt_pool, shuf_pool, mut_pool, del_pool, repl_pool])\
    .named('combo_pool')\
    .print_library()


template_pool: seq_length=51, num_states=1
state  seq
    0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc/>AGATCGGA

wt_pool: seq_length=5, num_states=3
state  name   seq
    0  wt.v0  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  wt.v1  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>aaaat</bc>AGATCGGA
    2  wt.v2  TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>gtggt</bc>AGATCGGA

combo_pool: seq_length=None, num_states=78
state  name              seq
    0  wt.v0             TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>tggcc</bc>AGATCGGA
    1  wt.v1             TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>ctgac</bc>AGATCGGA
    2  wt.v2             TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGGAAA</cre>ATTACGG<bc>caaaa</bc>AGATCGGA
    3  shuff_0.v0        TCCCGACT<cre>ggcgacaaccaaaggagggcggagaaat</cre>ATTACGG<bc/>AGATCGGA
    4  shuff_0.v1        TCCCGACT<cre>ggcgacaaccaaaggagggcggagaaat</cre

In [7]:
combo_pool.print_tree()

combo_pool (pool, n=78)
└── combo_pool.op [op=stack, mode=fixed, n=1]
    ├── wt_pool (pool, n=3)
    │   └── wt_pool.op [op=get_kmers, mode=random, n=1]
    │       └── pool[1] (pool, n=3)
    │           └── op[1]:repeat [op=repeat, mode=sequential, n=3]
    │               └── template_pool (pool, n=1)
    │                   └── template_pool.op [op=from_seq, mode=fixed, n=1]
    ├── shuf_pool (pool, n=15)
    │   └── shuf_pool.op [op=repeat, mode=sequential, n=3]
    │       └── pool[3] (pool, n=5)
    │           └── op[3]:shuffle_seq [op=shuffle_seq, mode=hybrid, n=5]
    │               └── template_pool (pool, n=1)
    │                   └── template_pool.op [op=from_seq, mode=fixed, n=1]
    ├── mut_rate_pool (pool, n=15)
    │   └── mut_rate_pool.op [op=get_kmers, mode=random, n=1]
    │       └── pool[6] (pool, n=15)
    │           └── op[6]:repeat [op=repeat, mode=sequential, n=3]
    │               └── pool[5] (pool, n=5)
    │                   └── op[5]:mutagenize [o

Pool(id=23, name='combo_pool', op='combo_pool.op', num_states=78)